# 层和块

## 自定义块

多层感知机（4.3小节）， nn.Sequential定义了一种很特殊的Module

torch.nn.functional（通常缩写为 F）是 PyTorch 中提供纯函数式神经网络操作的模块，包含各种无状态的数学运算和激活函数。

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[ 0.0160, -0.0004, -0.1903,  0.2982,  0.2557, -0.1218, -0.0699,  0.0308,
          0.0521,  0.2152],
        [-0.0633,  0.0132, -0.1872,  0.2119,  0.3536, -0.0394, -0.1092, -0.0880,
         -0.1404,  0.2083]], grad_fn=<AddmmBackward0>)

任何一个层或一个神经网络，他都是module的一个子类

In [ ]:
class MLP(nn.Module):  # 自定义继承nn.Module的类
    def __init__(self):
        super().__init__()  # 调用父类，内置参数全部置好
        self.hidden1 = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)
    
    def forward(self, X):
        return self.out(F.relu(self.hidden1(X)))  # 经过第一层函数之后通过激活函数relu，再经过第二层函数

In [4]:
net = MLP()
net(X)


tensor([[ 0.0498, -0.0225,  0.0865,  0.1302,  0.0315,  0.1345,  0.0944, -0.0969,
          0.0359, -0.0930],
        [-0.0316,  0.0028,  0.0911,  0.0582, -0.0336,  0.1078,  0.0657, -0.0852,
          0.0982, -0.1543]], grad_fn=<AddmmBackward0>)

## 视频的写法

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args): # *args 表示可以传入任意数量的参数
        super().__init__()
        for block in args:
            self._modules[block] = block # 创建了一个有序的字典，键是block，值是block，python3.6之后，字典都是有序字典
        
    def forward(self, X):
        for block in self._modules.values():  # 按顺序
            X = block(X)
        return X

这里调用net(X)没有报错，是因为MySequential类继承了nn.Module类，而nn.Module类实现了__call__方法

魔术方法__call__：在 Python 中，如果一个类定义了 __call__ 方法，那么这个类的实例（对象）就可以像普通的函数一样被“调用”；

如下代码：

```
class AddOne:
    def __init__(self):
        self.factor = 1
        
    def __call__(self, x):
        return x + self.factor

add_instance = AddOne()
print(add_instance(5))  # 输出 6，这里没有显式调用任何方法名，但背后自动触发了 __call__
```

MySequential 类继承了 nn.Module（通过 super().__init__() 进行了初始化）。在 PyTorch 官方的 nn.Module 基类源码中，已经默认实现好了 __call__ 方法,Python 检测到想把对象当函数用，于是调用了基类 nn.Module 中的 __call__(X),nn.Module.__call__ 内部会执行一系列 PyTorch 自带的安全检查和钩子函数,接着，它在内部执行了 result = self.forward(*input)。因为 Python 的多态性，这行代码会成功调用到你在 MySequential 中写好的自定义 forward(self, X)。

In [7]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.1294,  0.1183,  0.3545, -0.0881,  0.2197, -0.1694, -0.0736,  0.0678,
         -0.0948,  0.1312],
        [ 0.1340,  0.1785,  0.2881, -0.0122,  0.2466, -0.2981, -0.1409,  0.0626,
         -0.0971,  0.2575]], grad_fn=<AddmmBackward0>)

这里建议调用net(X), 而不是调用net.forward(X),如果直接调用 net.forward(X)，就会彻底绕过（Bypass） __call__ 方法，导致你注册的所有前向钩子（Forward Hooks）全部失效。这在提取中间层特征、可视化特征图、或者使用某些第三方调试库时会引发严重的 Bug。

## 顺序块

更加仔细的看一下Sequential如何工作的：

* 将块逐个追加到列表中的函数
* 一种前向传播函数，用于将输入按追加块的顺序传递给块组成的链条

In [8]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module  # 底层的module更希望key是字符串的数字类型

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [9]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)


tensor([[-0.1489,  0.2406,  0.0375, -0.1212,  0.1460, -0.2831, -0.0711,  0.0812,
         -0.0778, -0.1234],
        [ 0.0072,  0.0141, -0.0148, -0.1907,  0.1150, -0.2953, -0.2377,  0.1245,
         -0.1362,  0.0218]], grad_fn=<AddmmBackward0>)

## 在前向传播函数中执行代码

我们希望在前向传播的过程中执行python的控制流，下面的代码实现了一个计算函数$f(x, w) = cw^Tx$的层，其中x是输入，w是权重参数

In [11]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)  # 这个权重不是模型参数，永远不会被反向传播更新
        self.linear = nn.Linear(20, 20)

    def forward(self, X):  # 可以在forward里面写任何需要用到的函数
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [13]:
net = FixedHiddenMLP()
net(X)

tensor(0.1245, grad_fn=<SumBackward0>)

### 混合搭配各种组合块

矩阵的维度匹配就可以了

In [14]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.1263, grad_fn=<SumBackward0>)